In [1]:
!pip install playwright nest-asyncio pandas playwright-stealth
!playwright install chromium firefox webkit

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory



In [ ]:
# ============================================================
# CRAWL Agoda — v3 (API-first: nhanh & chính xác)
# ============================================================
# Luồng: đọc list KS từ CSV (Hotel, URL, Room) -> mở trang Agoda ->
#   BẮT THẲNG JSON API `/api/v1/property/room-grid` (thay vì scrape DOM) ->
#   lấy giá FINAL (all-in, vì URL có finalPriceView=1) của đúng room type -> 6 tuần.
#
# Vì sao nhanh hơn code cũ:
#   • Chờ ĐÚNG response API (không sleep/scroll mò) + chặn ảnh/media/font
#   • Crawl 6 tuần SONG SONG trong mỗi khách sạn
#   • Cắt các delay anti-detect quá liều
# Vì sao chính xác hơn:
#   • Dữ liệu JSON có cấu trúc (room.name + offers[].price.final)
#   • Match phòng theo TOKEN + ưu tiên khớp chính xác -> nhận diện đúng phòng
#     sold-out (room.isSoldOut), không khớp nhầm sang phòng khác
#   • Lấy đúng giá FINAL (sau giảm, đã gồm thuế/phí), chọn offer rẻ nhất

import asyncio
import nest_asyncio
import pandas as pd
import random
import time
import os
import re
import json
import glob
from datetime import datetime, timedelta
from playwright.async_api import async_playwright, TimeoutError as PWTimeout
from playwright_stealth import Stealth

nest_asyncio.apply()
_stealth = Stealth()

# ============================================================
# CONFIG
# ============================================================
# Tự tìm file input agoda*.csv trong thư mục hiện tại (bỏ qua TEMP_/FINAL_)
_csvs = sorted(f for f in glob.glob("*.csv") if not f.startswith(("TEMP_", "FINAL_")))
INPUT_FILE = _csvs[0] if _csvs else "./agoda1.csv"
TEMP_OUTPUT_FILE = "TEMP_" + os.path.basename(INPUT_FILE)
OUTPUT_PREFIX = "FINAL_"
INPUT_SHEET_NAME = "Hotel Link"        # chỉ dùng cho .xlsx
GOOGLE_SHEET_ID = ""                    # để trống = đọc file local
GOOGLE_SHEET_NAME = "Hotel Link"
COOKIES_FILE = "cookies.json"          # tuỳ chọn

# Loại giá ghi ra: final (all-in, mặc định — khớp dữ liệu cũ) | original (gạch ngang) | cashback
PRICE_TYPE = "final"

HEADLESS = True
CHECKIN_OFFSET = 3                      # W1 = hôm nay + 3 ngày
NUM_WEEKS = 6
DAYS_PER_WEEK = 3                       # số ngày thử/tuần (dừng khi có giá)
WEEKS_PARALLEL = 6                      # số tuần crawl song song trong 1 KS
PAGE_TIMEOUT = 40000
API_WAIT_TIMEOUT = 25                   # giây: chờ response room-grid
CURRENCY = "VND"

BETWEEN_HOTELS = (2.0, 5.0)            # nghỉ giữa các khách sạn (Agoda nhạy bot hơn)
NAV_JITTER = (0.3, 1.2)
INTRA_WEEK_DELAY = (0.5, 1.5)
AUTO_RETRY_NA_SOLDOUT = True
RETRY_DAYS_PER_WEEK = 5

ROOM_API_HINT = "/api/v1/property/room-grid"   # endpoint chứa list phòng + giá
BLOCK_RESOURCE_TYPES = {"image", "media", "font"}

USER_AGENTS = [
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]
SCREEN_RESOLUTIONS = [(1366, 768), (1440, 900), (1536, 864), (1600, 900), (1920, 1080), (1680, 1050)]

# ============================================================
# EXTRACTION (đã test với dữ liệu Agoda thật)
# ============================================================
def _to_int(s):
    n = re.sub(r'[^\d]', '', str(s) if s is not None else '')
    return int(n) if n else None

def parse_amount(text):
    if text is None:
        return None
    m = re.search(r'([\d][\d.,]{3,})', str(text))
    if not m:
        return None
    v = _to_int(m.group(1))
    return v if v and v >= 1000 else None

def fmt_price(v):
    return f"{v:,}"                     # Agoda format: số thuần, KHÔNG prefix "VND" (khớp dữ liệu cũ)

def _norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', (s or '').lower()).strip()

def _tokens(s):
    return set(t for t in _norm(s).split() if t)

def _best_room(rooms, target):
    """Khớp target với 1 room dict (gồm cả phòng sold-out). Ưu tiên khớp CHÍNH XÁC
    -> tránh khớp nhầm sang phòng khác khi phòng đúng đang sold-out."""
    named = [(rm, (rm.get("name") or "").strip()) for rm in rooms]
    named = [(rm, n) for rm, n in named if n]
    if not named:
        return None
    tnorm = _norm(target)
    ttok = _tokens(target)
    for rm, n in named:                                  # 1) khớp chính xác
        if _norm(n) == tnorm:
            return rm
    tfirst = tnorm.split()[0] if tnorm else ""
    best, best_score = None, -1.0                        # 2) Jaccard + bonus trùng từ đầu
    for rm, n in named:
        ntok = _tokens(n)
        if not ntok:
            continue
        score = len(ttok & ntok) / max(len(ttok | ntok), 1)
        nfirst = _norm(n).split()[0] if _norm(n) else ""
        if tfirst and tfirst == nfirst:
            score += 0.3
        if score > best_score:
            best_score, best = score, rm
    return best if best_score >= 0.5 else None

def agoda_offer_price(offer, ptype=PRICE_TYPE):
    pr = (offer or {}).get("price") or {}
    if ptype == "cashback":
        node = (pr.get("cashback") or {}).get("price") or {}
    else:
        node = pr.get(ptype) or {}                       # "final" | "original"
    v = node.get("amountNumber")
    if isinstance(v, (int, float)) and v >= 1000:
        return int(v)
    return parse_amount(node.get("amount") or node.get("text"))

def extract_from_agoda(rg, target_room, ptype=PRICE_TYPE):
    rooms = (rg or {}).get("rooms") or []
    rm = _best_room(rooms, target_room)
    if rm is not None:
        prices = [agoda_offer_price(o, ptype) for o in (rm.get("offers") or [])]
        prices = [p for p in prices if p]
        if prices:
            return {"found": True, "price": fmt_price(min(prices)), "room": rm.get("name")}
        if rm.get("isSoldOut") or not rm.get("offers"):
            return {"found": False, "soldOut": True, "room": rm.get("name")}
    if rg.get("isSoldOut"):
        return {"found": False, "soldOut": True}
    return {"found": False, "soldOut": False, "rooms": [(r.get("name") or "") for r in rooms]}

def is_real(v):
    return v not in (None, "", "NA", "nan") and not str(v).startswith("SOLD OUT")

# ============================================================
# INPUT READERS
# ============================================================
def read_hotels_from_source():
    if GOOGLE_SHEET_ID:
        return read_hotels_from_gsheet(GOOGLE_SHEET_ID, GOOGLE_SHEET_NAME)
    if INPUT_FILE.endswith('.xlsx'):
        return read_hotels_from_xlsx(INPUT_FILE, INPUT_SHEET_NAME)
    return read_hotels_from_csv(INPUT_FILE)

def _norm_cols(df):
    req = ['hotel_name', 'hotel_url', 'room_type']
    if not all(c in df.columns for c in req):
        df.columns = req + list(df.columns[3:])
    df = df[df['hotel_url'].notna() & (df['hotel_url'] != '')]
    return df[['hotel_name', 'hotel_url', 'room_type']]

def read_hotels_from_csv(file_path):
    try:
        df = _norm_cols(pd.read_csv(file_path))
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df
    except Exception as e:
        print(f"❌ Lỗi đọc CSV: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_xlsx(file_path, sheet_name):
    try:
        from openpyxl import load_workbook   # import lazy (cell install không có openpyxl)
        wb = load_workbook(filename=file_path, data_only=True)
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(min_row=2, max_col=2):
            hcell, rcell = row[0], row[1]
            url = hcell.hyperlink.target if hcell.hyperlink else (hcell.value or "")
            rows.append({'hotel_name': hcell.value or "", 'hotel_url': url,
                         'room_type': (rcell.value if rcell else "")})
        df = pd.DataFrame(rows)
        df = df[df['hotel_url'].notna() & df['hotel_url'].astype(str).str.startswith('http')]
        print(f"✅ {len(df)} hotels từ {file_path}", flush=True)
        return df[['hotel_name', 'hotel_url', 'room_type']]
    except Exception as e:
        print(f"❌ Lỗi đọc xlsx: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

def read_hotels_from_gsheet(sheet_id, sheet_name=""):
    from urllib.parse import quote
    try:
        url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv"
        if sheet_name:
            url += f"&sheet={quote(sheet_name)}"
        df = _norm_cols(pd.read_csv(url))
        print(f"✅ {len(df)} hotels từ Google Sheet", flush=True)
        return df
    except Exception as e:
        print(f"❌ Lỗi đọc Google Sheet: {e}", flush=True)
        return pd.DataFrame(columns=['hotel_name', 'hotel_url', 'room_type'])

# ============================================================
# URL & SAVE
# ============================================================
def update_url_checkin(url, checkin):
    """Agoda dùng checkin=YYYY-MM-DD + los=1 (1 đêm), KHÔNG có checkout."""
    ci = checkin.strftime("%Y-%m-%d")
    if re.search(r'check[Ii]n=', url):
        url = re.sub(r'check[Ii]n=[\d-]+', f'checkin={ci}', url)
    else:
        url += f"{'&' if '?' in url else '?'}checkin={ci}"
    if 'los=' not in url.lower():
        url += '&los=1'
    if 'currencycode=' not in url.lower():
        url += f'&currencyCode={CURRENCY}'
    if 'finalpriceview=' not in url.lower():      # đảm bảo giá all-in
        url += '&finalPriceView=1'
    return url

def save_backup_csv(all_week_prices, filename):
    """Lưu CSV — only-improve (không ghi NA đè giá thật) + atomic write."""
    try:
        rows, written = [], set()
        if os.path.exists(filename):
            try:
                df_old = pd.read_csv(filename, keep_default_na=False, na_values=[])
                for _, row in df_old.iterrows():
                    k = (str(row.get("hotel_name", "")), str(row.get("room_type", "")))
                    written.add(k)
                    if k in all_week_prices:
                        nr = {"hotel_name": k[0], "room_type": k[1]}
                        for i in range(1, NUM_WEEKS + 1):
                            old = str(row.get(f"price_w{i}", "NA")).strip()
                            new = str(all_week_prices[k].get(f"Price W{i}", "NA")).strip()
                            if new in ("NA", "nan", "") and old not in ("NA", "nan", ""):
                                nr[f"price_w{i}"] = old
                            else:
                                nr[f"price_w{i}"] = new if new not in ("nan", "") else "NA"
                        rows.append(nr)
                    else:
                        rows.append(row.to_dict())
            except Exception:
                pass
        for (hotel, room), prices in all_week_prices.items():
            if (hotel, room) not in written:
                r = {"hotel_name": hotel, "room_type": room}
                for i in range(1, NUM_WEEKS + 1):
                    r[f"price_w{i}"] = prices.get(f"Price W{i}", "NA")
                rows.append(r)
        tmp = filename + ".tmp"
        pd.DataFrame(rows).to_csv(tmp, index=False)
        os.replace(tmp, filename)
    except Exception as e:
        print(f"❌ Error saving: {e}", flush=True)

async def load_cookies_to_context(context):
    if not os.path.exists(COOKIES_FILE):
        return
    try:
        with open(COOKIES_FILE, "r", encoding="utf-8") as f:
            cookies = json.load(f)
        pw = []
        for c in cookies:
            ck = {"name": c.get("name", ""), "value": c.get("value", ""),
                  "domain": c.get("domain", ".agoda.com"), "path": c.get("path", "/")}
            for opt in ("expires", "httpOnly", "secure"):
                if c.get(opt):
                    ck[opt] = c[opt]
            if c.get("sameSite") in ("Strict", "Lax", "None"):
                ck["sameSite"] = c["sameSite"]
            pw.append(ck)
        await context.add_cookies(pw)
    except Exception:
        pass

# ============================================================
# BROWSER / CRAWL
# ============================================================
async def _block_route(route):
    try:
        if route.request.resource_type in BLOCK_RESOURCE_TYPES:
            await route.abort()
        else:
            await route.continue_()
    except Exception:
        try:
            await route.continue_()
        except Exception:
            pass

async def make_context(browser):
    res = random.choice(SCREEN_RESOLUTIONS)
    ctx = await browser.new_context(
        viewport={"width": res[0], "height": res[1]},
        user_agent=random.choice(USER_AGENTS),
        locale="en-GB",
        timezone_id="Asia/Ho_Chi_Minh",
        java_script_enabled=True,
    )
    await ctx.route("**/*", _block_route)
    await load_cookies_to_context(ctx)
    return ctx

async def crawl_day(ctx, hotel_url, room_type, checkin, page_timeout=PAGE_TIMEOUT):
    """Mở 1 ngày check-in, bắt JSON API room-grid -> giá. (Agoda: API-only;
    nếu không bắt được API thì trả NA và để vòng retry thử lại.)"""
    page = await ctx.new_page()
    captured = {}

    def on_resp(resp):
        if "api_resp" in captured:
            return
        try:
            if ROOM_API_HINT in resp.url and resp.status == 200:
                captured["api_resp"] = resp
        except Exception:
            pass

    page.on("response", on_resp)
    try:
        await _stealth.apply_stealth_async(page)
        url = update_url_checkin(hotel_url, checkin)
        await asyncio.sleep(random.uniform(*NAV_JITTER))
        try:
            await page.goto(url, timeout=page_timeout, wait_until="domcontentloaded")
        except Exception:
            pass

        waited = 0.0
        while "api_resp" not in captured and waited < API_WAIT_TIMEOUT:
            await asyncio.sleep(0.25)
            waited += 0.25

        if "api_resp" in captured:
            try:
                rg = await captured["api_resp"].json()
                return extract_from_agoda(rg, room_type)
            except Exception:
                pass
        return {"found": False, "soldOut": False}
    finally:
        try:
            await page.close()
        except Exception:
            pass

async def crawl_week(ctx, hotel_url, room_type, week_num, base_checkin,
                     hotel_name="", days=DAYS_PER_WEEK, page_timeout=PAGE_TIMEOUT):
    week_start = base_checkin + timedelta(days=(week_num - 1) * 7)
    sold = False
    for d in range(days):
        checkin = week_start + timedelta(days=d)
        res = await crawl_day(ctx, hotel_url, room_type, checkin, page_timeout)
        if res.get("found"):
            print(f"      ✅ [{hotel_name[:22]}] W{week_num}: {res['price']} "
                  f"({checkin.strftime('%m/%d')}, {res.get('room','')[:20]})", flush=True)
            return {"week": week_num, "price": res["price"]}
        if res.get("soldOut"):
            sold = True
        if d < days - 1:
            await asyncio.sleep(random.uniform(*INTRA_WEEK_DELAY))
    price = "SOLD OUT" if sold else "NA"
    print(f"      {'🚫' if sold else '❌'} [{hotel_name[:22]}] W{week_num}: {price}", flush=True)
    return {"week": week_num, "price": price}

async def process_hotel(browser, info, prev, base_checkin, awp):
    hotel_name, hotel_url, room_type = info
    key = (hotel_name, room_type)

    prices = {f"Price W{i}": "NA" for i in range(1, NUM_WEEKS + 1)}
    if key in prev:
        for i in range(1, NUM_WEEKS + 1):
            v = prev[key].get(f"Price W{i}", "NA")
            if is_real(v):
                prices[f"Price W{i}"] = v

    weeks = [i for i in range(1, NUM_WEEKS + 1) if not is_real(prices[f"Price W{i}"])]
    if not weeks:
        awp[key] = prices
        return key, prices, True

    print(f"\n🏨 {hotel_name} | {room_type} | tuần cần crawl: {weeks}", flush=True)
    ctx = await make_context(browser)
    try:
        sem = asyncio.Semaphore(WEEKS_PARALLEL)

        async def one(wn):
            async with sem:
                return await crawl_week(ctx, hotel_url, room_type, wn, base_checkin, hotel_name)

        results = await asyncio.gather(*[one(w) for w in weeks])
    finally:
        try:
            await ctx.close()
        except Exception:
            pass

    for r in results:
        prices[f"Price W{r['week']}"] = r["price"]
    awp[key] = prices
    na = sum(1 for i in range(1, NUM_WEEKS + 1) if prices[f"Price W{i}"] == "NA")
    so = sum(1 for i in range(1, NUM_WEEKS + 1) if str(prices[f"Price W{i}"]).startswith("SOLD OUT"))
    icon = "✅" if na == 0 and so == 0 else (f"⚠️({na}NA)" if na else f"🚫({so}SO)")
    print(f"   {icon} DONE: {hotel_name}", flush=True)
    return key, prices, False

# ============================================================
# MAIN
# ============================================================
async def main():
    t0 = time.time()
    df = read_hotels_from_source()
    if len(df) == 0:
        return

    awp, prev = {}, {}

    def load_prev(fp):
        n = 0
        try:
            dp = pd.read_csv(fp, keep_default_na=False, na_values=[])
            for _, row in dp.iterrows():
                k = (row["hotel_name"], row["room_type"])
                p = {}
                for i in range(1, NUM_WEEKS + 1):
                    v = str(row.get(f"price_w{i}", "NA")).strip()
                    p[f"Price W{i}"] = "NA" if (not v or v in ("nan", "NA")) else v
                prev[k] = p
                awp[k] = dict(p)
                n += 1
        except Exception:
            pass
        return n

    if os.path.exists(TEMP_OUTPUT_FILE):
        print(f"📂 Loaded {load_prev(TEMP_OUTPUT_FILE)} hotels từ temp", flush=True)

    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=CHECKIN_OFFSET)
    all_infos = [(r['hotel_name'], r['hotel_url'], r['room_type']) for _, r in df.iterrows()]

    done = sum(1 for info in all_infos
               if (info[0], info[2]) in prev
               and all(is_real(prev[(info[0], info[2])].get(f"Price W{i}", "NA")) for i in range(1, NUM_WEEKS + 1)))

    print(f"\n📅 W1 = {bc.strftime('%Y-%m-%d (%a)')} ... W{NUM_WEEKS} (mỗi tuần thử {DAYS_PER_WEEK} ngày)", flush=True)
    print(f"{'='*60}", flush=True)
    print(f"🎭 CRAWL Agoda v3 — API-first ({ROOM_API_HINT}) | giá: {PRICE_TYPE}", flush=True)
    print(f"📊 {len(all_infos)} hotels ({done} đã đủ 6w) | {WEEKS_PARALLEL} tuần song song/KS", flush=True)
    print(f"{'='*60}", flush=True)

    launch_args = ['--disable-blink-features=AutomationControlled', '--no-sandbox', '--disable-dev-shm-usage']
    if HEADLESS:
        launch_args.insert(0, '--headless=new')

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS, args=launch_args)
        try:
            for idx, info in enumerate(all_infos, 1):
                try:
                    key, prices, skip = await process_hotel(browser, info, prev, bc, awp)
                    if not skip:
                        save_backup_csv(awp, TEMP_OUTPUT_FILE)
                        await asyncio.sleep(random.uniform(*BETWEEN_HOTELS))
                except Exception as e:
                    print(f"❌ {info[0]}: {e}", flush=True)
                print(f"   ⏱️ {idx}/{len(all_infos)} | {int((time.time()-t0)//60)}m", flush=True)

            fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
            save_backup_csv(awp, fn)
            tc = len(awp) * NUM_WEEKS
            na = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if pp.get(f"Price W{i}", "NA") == "NA")
            so = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if str(pp.get(f"Price W{i}", "")).startswith("SOLD OUT"))
            print(f"\n📁 Round 1: {fn} | ✅ {tc-na-so}/{tc} | 🚫 {so} SO | ❌ {na} NA", flush=True)

            # ----- VÒNG 2: re-crawl NA / SOLD OUT -----
            if AUTO_RETRY_NA_SOLDOUT and (na > 0 or so > 0):
                ki = {(i[0], i[2]): i for i in all_infos}
                retry = {k: [i for i in range(1, NUM_WEEKS + 1)
                             if not is_real(awp[k].get(f"Price W{i}", "NA"))]
                         for k in awp if any(not is_real(awp[k].get(f"Price W{i}", "NA")) for i in range(1, NUM_WEEKS + 1))}
                retry = {k: v for k, v in retry.items() if v and k in ki}
                print(f"\n🔄 VÒNG 2: {sum(len(v) for v in retry.values())} ô / {len(retry)} KS", flush=True)
                updated = 0
                for k, weeks in retry.items():
                    hn, hu, rt = ki[k]
                    ctx = await make_context(browser)
                    try:
                        sem = asyncio.Semaphore(WEEKS_PARALLEL)

                        async def one(wn):
                            async with sem:
                                return await crawl_week(ctx, hu, rt, wn, bc, hn,
                                                        days=RETRY_DAYS_PER_WEEK, page_timeout=50000)
                        res = await asyncio.gather(*[one(w) for w in weeks])
                    finally:
                        try:
                            await ctx.close()
                        except Exception:
                            pass
                    for r in res:
                        old = awp[k].get(f"Price W{r['week']}", "NA")
                        new = r["price"]
                        if is_real(new) and new != old:
                            awp[k][f"Price W{r['week']}"] = new
                            updated += 1
                        elif new == "SOLD OUT" and old == "NA":
                            awp[k][f"Price W{r['week']}"] = new
                    save_backup_csv(awp, TEMP_OUTPUT_FILE)
                    await asyncio.sleep(random.uniform(*BETWEEN_HOTELS))
                save_backup_csv(awp, fn)
                print(f"   🔄 Vòng 2 xong: cập nhật {updated} ô", flush=True)
        finally:
            try:
                await browser.close()
            except Exception:
                pass

    tt = time.time() - t0
    tc = len(awp) * NUM_WEEKS
    na = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if pp.get(f"Price W{i}", "NA") == "NA")
    so = sum(1 for pp in awp.values() for i in range(1, NUM_WEEKS + 1) if str(pp.get(f"Price W{i}", "")).startswith("SOLD OUT"))
    fn = f"{OUTPUT_PREFIX}{datetime.today().strftime('%Y%m%d')}.csv"
    print(f"\n{'='*60}", flush=True)
    print(f"✅ HOÀN TẤT | {fn}", flush=True)
    print(f"   ✅ Giá: {tc-na-so}/{tc} ({(tc-na-so)/max(tc,1):.1%}) | 🚫 {so} SO | ❌ {na} NA", flush=True)
    print(f"⏱️ {int(tt//60)}m {int(tt%60)}s", flush=True)
    print(f"{'='*60}", flush=True)

await main()


✅ 30 hotels từ agoda4.csv
📂 Loaded 30 hotels từ temp

📅 W1 = 2026-06-27 (Sat) ... W6 (mỗi tuần thử 3 ngày)
🎭 CRAWL Agoda v3 — API-first (/api/v1/property/room-grid) | giá: final
📊 30 hotels (13 đã đủ 6w) | 6 tuần song song/KS
📂 Loaded 30 hotels từ temp

📅 W1 = 2026-06-27 (Sat) ... W6 (mỗi tuần thử 3 ngày)
🎭 CRAWL Agoda v3 — API-first (/api/v1/property/room-grid) | giá: final
📊 30 hotels (13 đã đủ 6w) | 6 tuần song song/KS
   ⏱️ 1/30 | 0m
   ⏱️ 2/30 | 0m

🏨 The Mist Boutique - Executive Suite | Executive Suite | tuần cần crawl: [5]
      🚫 [The Mist Boutique - Ex] W5: SOLD OUT
   🚫(1SO) DONE: The Mist Boutique - Executive Suite
   ⏱️ 3/30 | 0m
   ⏱️ 4/30 | 0m
   ⏱️ 5/30 | 0m

🏨 Mai Chau Ecolodge | Superior Twin Room with Garden View | tuần cần crawl: [1, 2, 3, 4, 5, 6]
      ✅ [Mai Chau Ecolodge] W6: 2,328,042 (08/01, Superior Twin Room w)
      ✅ [Mai Chau Ecolodge] W1: 2,031,746 (06/28, Superior Double Bed )
      ✅ [Mai Chau Ecolodge] W3: 2,074,074 (07/12, Superior Twin Room w)
     